## Import Needed Package

In [1]:
import pandas as pd
import numpy as np
import warnings 
warnings.filterwarnings("ignore")

## Load Dataset

In [2]:
data = pd.read_csv("./IMDB_Dataset.csv")

In [3]:
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
data["sentiment"].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

# One Hot Encoding

## Label Encoder

In [5]:
data.replace({"sentiment":{"positive": 1, "negative": 0}}, inplace=True)

In [6]:
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


# Data Preprocessing

In [9]:
pip install tensorflow

Note: you may need to restart the kernel to use updated packages.


In [11]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [12]:
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

In [14]:
train_data.shape

(40000, 2)

In [15]:
test_data.shape

(10000, 2)

In [16]:
tokenizer = Tokenizer(num_words = 5000)
tokenizer.fit_on_texts(train_data["review"])

In [23]:
X_train = pad_sequences(tokenizer.texts_to_sequences(train_data["review"]), maxlen=200)
X_test = pad_sequences(tokenizer.texts_to_sequences(test_data["review"]), maxlen=200)

In [24]:
X_train

array([[1935,    1, 1200, ...,  205,  351, 3856],
       [   3, 1651,  595, ...,   89,  103,    9],
       [   0,    0,    0, ...,    2,  710,   62],
       ...,
       [   0,    0,    0, ..., 1641,    2,  603],
       [   0,    0,    0, ...,  245,  103,  125],
       [   0,    0,    0, ...,   70,   73, 2062]], dtype=int32)

In [25]:
X_test

array([[   0,    0,    0, ...,  995,  719,  155],
       [  12,  162,   59, ...,  380,    7,    7],
       [   0,    0,    0, ...,   50, 1088,   96],
       ...,
       [   0,    0,    0, ...,  125,  200, 3241],
       [   0,    0,    0, ..., 1066,    1, 2305],
       [   0,    0,    0, ...,    1,  332,   27]], dtype=int32)

In [26]:
Y_train = train_data["sentiment"]
Y_test = test_data["sentiment"]

In [27]:
Y_train

39087    0
30893    0
45278    1
16398    0
13653    0
        ..
11284    1
44732    1
38158    0
860      1
15795    1
Name: sentiment, Length: 40000, dtype: int64

In [28]:
Y_test

33553    1
9427     1
199      0
12447    1
39489    0
        ..
28567    0
25079    1
18707    1
15200    0
5857     1
Name: sentiment, Length: 10000, dtype: int64

# Model Building

In [30]:
model = Sequential()
model.add(Embedding(input_dim = 5000, output_dim = 128, input_length = 200))
model.add(LSTM(128, dropout=0.2, recurrent_dropout = 0.2))
model.add(Dense(1, activation = "sigmoid"))

In [31]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [32]:
model.compile(optimizer = "adam",loss="binary_crossentropy", metrics=["accuracy"])

In [33]:
model.fit(X_train, Y_train, epochs = 5, batch_size = 64, validation_split = 0.2)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 354s 700ms/step - accuracy: 0.7857 - loss: 0.4635 - val_accuracy: 0.8508 - val_loss: 0.3502
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 372s 744ms/step - accuracy: 0.8351 - loss: 0.3924 - val_accuracy: 0.8443 - val_loss: 0.3623
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 362s 723ms/step - accuracy: 0.8712 - loss: 0.3170 - val_accuracy: 0.8618 - val_loss: 0.3345
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 359s 717ms/step - accuracy: 0.8881 - loss: 0.2838 - val_accuracy: 0.8741 - val_loss: 0.3085
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 369s 737ms/step - accuracy: 0.9052 - loss: 0.2444 - val_accuracy: 0.8749 - val_loss: 0.3361


In [34]:
loss, accuracy = model.evaluate (X_test, Y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 23s 71ms/step - accuracy: 0.8774 - loss: 0.3338


In [35]:
print(loss)

0.3338111937046051


In [36]:
print(accuracy)

0.8773999810218811


# Building Predictive System

In [37]:
def predictive_system(review):
    sequences = tokenizer.texts_to_sequences([review])
    padded_sequence = pad_sequences(sequences, maxlen=200)
    prediction = model.predict(padded_sequence)
    sentiment = "positive" if prediction[0][0] > 0.5 else "negative"
    return sentiment

In [38]:
predictive_system("This movie was fantastic and amazing")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 632ms/step


'positive'

In [39]:
predictive_system("A trilling adventure with stunning visual")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step


'positive'

In [40]:
predictive_system("A visual masterpiece")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step


'positive'

In [41]:
predictive_system("Overall long and slow")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step


'negative'

# Saving Model

In [42]:
model.save("model.h5")

In [43]:
import joblib
joblib.dump(tokenizer, "tokenizer.pkl")

['tokenizer.pkl']